In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import numpy as np
import pandas as pd
df=pd.read_csv("/kaggle/input/datasets/adithip2000/genre-generalised-and-tagged/final_processed_data.csv")

In [ ]:
!pip install iterative-stratification

In [ ]:
df.drop(columns=['Unnamed: 0'],inplace=True)

In [ ]:
df.head()

In [ ]:
from transformers import AutoTokenizer,AutoModel
import torch
model_id="roberta-base"
tokenizer=AutoTokenizer.from_pretrained(model_id)
#model = AutoModel.from_pretrained(model_id)
#device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
#model.to(device)


Converting genre to list then to multihot vector

In [ ]:
import ast
type(df['genres'].iloc[0])
df['genres']=df['genres'].apply(ast.literal_eval)

type(df['genres'].iloc[0])



In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer
mlb=MultiLabelBinarizer()
y=mlb.fit_transform(df['genres'])
print(y[0])

In [ ]:
y.shape

In [ ]:
label_counts=np.sum(y,axis=0)
for genre, count in zip(mlb.classes_, label_counts):
    print(genre, count)

Train Test Split

In [ ]:
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

X = df['summary'].reset_index(drop=True)
y = y  

# 🔹 First split: train vs temp (80/20)
msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

for train_idx, temp_idx in msss.split(X, y):
    X_train, X_temp = X.iloc[train_idx], X.iloc[temp_idx]
    y_train, y_temp = y[train_idx], y[temp_idx]

# 🔹 Second split: val vs test (10/10)
msss2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=42)

for val_idx, test_idx in msss2.split(X_temp, y_temp):
    X_val, X_test = X_temp.iloc[val_idx], X_temp.iloc[test_idx]
    y_val, y_test = y_temp[val_idx], y_temp[test_idx]

In [ ]:
print(y_train.sum(axis=0))
print(y_val.sum(axis=0))
print(y_test.sum(axis=0))

In [ ]:
import torch
from torch.utils.data import Dataset

class GenreDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, vectorizer, max_len=128):
        """
        texts: pandas Series of text
        labels: multi-hot numpy array or list
        tokenizer: HuggingFace tokenizer
        vectorizer: fitted CountVectorizer / TfidfVectorizer
        """

        self.texts = texts.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len

        
        self.labels = torch.tensor(labels, dtype=torch.float32)

        
        bow_matrix = vectorizer.transform(self.texts)
        self.bow = torch.tensor(bow_matrix.toarray(), dtype=torch.float32)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]

        #  Tokenize
        encoding = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_len,
            return_tensors='pt'
        )

        #  Remove batch dimension
        input_ids = encoding['input_ids'].squeeze(0)
        attention_mask = encoding['attention_mask'].squeeze(0)

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "bow": self.bow[idx],
            "labels": self.labels[idx]
        }

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel

class TopicGenreModel(nn.Module):
    def __init__(self, num_topics, num_genres, vocab_size, model_name="roberta-base"):
        super().__init__()

        # 🔹 Backbone (RoBERTa)
        self.backbone = AutoModel.from_pretrained(model_name)
        bert_dim = self.backbone.config.hidden_size

        # 🔹 Context encoder ONLY (no BoW encoder)
        self.encoder = nn.Sequential(
            nn.Linear(bert_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2)
        )

        # 🔹 Latent space
        self.fc_mu = nn.Linear(256, num_topics)
        self.fc_logvar = nn.Linear(256, num_topics)

        # 🔹 Topic → word matrix
        self.beta = nn.Parameter(torch.randn(num_topics, vocab_size))
        nn.init.xavier_uniform_(self.beta)

        # 🔹 Topic → genre classifier
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(num_topics, num_genres)
        )

    # ---------- Encode ----------
    def encode(self, embedding):
        h = self.encoder(embedding)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar

    # ---------- Reparameterization ----------
    def reparameterize(self, mu, logvar):
        logvar = torch.clamp(logvar, -10, 10)  # stability
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    # ---------- Decode ----------
    def decode(self, topic_dist):
        beta_norm = torch.softmax(self.beta, dim=1)  
        logits = torch.matmul(topic_dist, beta_norm)
        return torch.log_softmax(logits, dim=1)

    # ---------- Forward ----------
    def forward(self, input_ids, attention_mask):

        # 🔹 Get contextual embedding (MEAN pooling)
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)

        mask = attention_mask.unsqueeze(-1)
        embedding = (outputs.last_hidden_state * mask).sum(dim=1) / mask.sum(dim=1)

        # 🔹 Encode → latent topics
        mu, logvar = self.encode(embedding)

        # 🔹 Sample
        z = self.reparameterize(mu, logvar)
        topic_dist = torch.softmax(z, dim=1)

        # 🔹 Genre prediction
        genre_logits = self.classifier(topic_dist)

        # 🔹 Word reconstruction
        word_log_probs = self.decode(topic_dist)

        return topic_dist, genre_logits, word_log_probs, mu, logvar

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer(
    max_features=8000,
    stop_words='english',
    min_df=5,
    max_df=0.8,
    ngram_range=(1,1)  # keep simple for now
)

vectorizer.fit(X_train.values)

In [ ]:
train_dataset = GenreDataset(
    texts=X_train,
    labels=y_train,
    tokenizer=tokenizer,
    vectorizer=vectorizer,
    max_len=128
)

val_dataset = GenreDataset(
    texts=X_val,
    labels=y_val,
    tokenizer=tokenizer,
    vectorizer=vectorizer,
    max_len=128
)

test_dataset = GenreDataset(
    texts=X_test,
    labels=y_test,
    tokenizer=tokenizer,
    vectorizer=vectorizer,
    max_len=128
)

In [ ]:
sample = train_dataset[0]

print(sample["input_ids"].shape)     # [256]
print(sample["attention_mask"].shape)
print(sample["bow"].shape)           # [8000]
print(sample["labels"].shape)        # [num_genres]

In [ ]:
from torch.utils.data import DataLoader

# 🔹 Vocabulary size 
vocab_size = len(vectorizer.get_feature_names_out())

# 🔹 Optional sanity check
bow_dim = train_dataset.bow.shape[1]
print("Vocab size:", vocab_size, "| Bow dim:", bow_dim)

# 🔹 Device-aware settings
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 🔹 DataLoader settings
BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,        
    pin_memory=True       
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [ ]:
def reconstruction_loss(word_log_probs, bow):
    bow_norm = bow / (bow.sum(dim=1, keepdim=True) + 1e-6)
    return -(bow_norm * word_log_probs).sum(dim=1).mean()


def kl_divergence(mu, logvar):
    return -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1).mean()
    
# class_weights = total_samples / (len(label_counts) * label_counts)
label_counts = np.sum(y_train, axis=0)
total_samples = y_train.shape[0]
num_pos=label_counts
num_neg=total_samples-num_pos
class_weights=num_neg/(num_pos+1e-6)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

classification_loss_fn = nn.BCEWithLogitsLoss(pos_weight=class_weights)  



In [ ]:
def loss_function(word_log_probs, bow, genre_logits, labels, mu, logvar,
                  alpha=0.0, beta=0.0, recon=1.0):

    recon_loss = reconstruction_loss(word_log_probs, bow)
    kl_loss = kl_divergence(mu, logvar)
    cls_loss = classification_loss_fn(genre_logits, labels)

    loss = (alpha * cls_loss) + (recon * recon_loss) + (beta * kl_loss)

    return loss, cls_loss, recon_loss, kl_loss

Training

In [ ]:
def train_one_epoch(model, loader, device, loss_fn, optimizer, epoch, num_epochs):
    model.train()

    total_loss = 0
    total_cls = 0
    total_recon = 0
    total_kl = 0

    #  KL annealing
    #beta = min(1.0, epoch / 10)
    if epoch < 2:
     beta = 0.0
     alpha = 0.0
    elif epoch < 10:
     beta = (epoch-2) / 50
    else:
     beta = 0.01

    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        bow = batch['bow'].to(device)
        labels = batch['labels'].to(device)

        # 🔹 Forward (NO bow here)
        topic_dist, genre_logits, word_log_probs, mu, logvar = model(
            input_ids, attention_mask
        )

        # 🔹 Loss
        loss, cls_loss, recon_loss, kl_loss = loss_fn(
            word_log_probs,
            bow,
            genre_logits,
            labels,
            mu,
            logvar,
            alpha=1.0,
            beta=beta,
            recon=0.5
        )

        optimizer.zero_grad()
        loss.backward()

        # 🔹 Gradient clipping
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()

        total_loss += loss.item()
        total_cls += cls_loss.item()
        total_recon += recon_loss.item()
        total_kl += kl_loss.item()

    # 🔹 Averages
    avg_loss = total_loss / len(loader)
    avg_cls = total_cls / len(loader)
    avg_recon = total_recon / len(loader)
    avg_kl = total_kl / len(loader)

    return avg_loss, avg_cls, avg_recon, avg_kl

In [ ]:
from torchmetrics.classification import MultilabelAccuracy, MultilabelF1Score

def validation(model, loader, device, loss_fn, num_labels, threshold=0.3):

    model.eval()

    total_loss = 0
    total_r = 0
    total_kl = 0
    total_cl = 0

    # 🔹 Metrics
    acc_metric = MultilabelAccuracy(num_labels=num_labels, threshold=threshold).to(device)
    f1_macro = MultilabelF1Score(num_labels=num_labels, average='macro', threshold=threshold).to(device)
    f1_weighted = MultilabelF1Score(num_labels=num_labels, average='weighted', threshold=threshold).to(device)

    acc_metric.reset()
    f1_macro.reset()
    f1_weighted.reset()

    with torch.no_grad():
        for batch in loader:

            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            bow = batch['bow'].to(device)
            labels = batch['labels'].to(device).float()

            # 🔹 Forward (FIXED)
            topic_dist, genre_logits, word_log_probs, mu, logvar = model(
                input_ids, attention_mask
            )

            # 🔹 Loss (FIXED unpacking)
            loss, cls_loss, recon_loss, kl_loss = loss_fn(
                word_log_probs,
                bow,
                genre_logits,
                labels,
                mu,
                logvar,
                alpha=1.0,
                beta=1.0,     #  during validation use full KL
                recon=0.1
            )

            total_loss += loss.item()
            total_r += recon_loss.item()
            total_kl += kl_loss.item()
            total_cl += cls_loss.item()

            # 🔹 Predictions (let torchmetrics handle threshold)
            probs = torch.sigmoid(genre_logits)

            acc_metric.update(probs, labels.int())
            f1_macro.update(probs, labels.int())
            f1_weighted.update(probs, labels.int())

    # 🔹 Averages
    avg_loss = total_loss / len(loader)
    avg_r = total_r / len(loader)
    avg_kl = total_kl / len(loader)
    avg_cl = total_cl / len(loader)

    # 🔹 Metrics
    val_acc = acc_metric.compute().item()
    val_f1_macro = f1_macro.compute().item()
    val_f1_weighted = f1_weighted.compute().item()

    return avg_loss, val_acc, val_f1_macro, val_f1_weighted, avg_r, avg_kl, avg_cl

In [ ]:


num_genres = len(mlb.classes_)
num_topics = 32

print(f"Vocab size: {vocab_size}")
print(f"num_genres: {num_genres}")
print(f"num_topics: {num_topics}")

# 🔹 Model init (FIXED)
model = TopicGenreModel(
    num_topics=num_topics,
    num_genres=num_genres,
    vocab_size=vocab_size,
    model_name="roberta-base"
)

model = model.to(device)


In [ ]:
for param in model.backbone.parameters():
    param.requires_grad = False
    optimizer = torch.optim.Adam(
    model.parameters(),
    lr=2e-4,
    eps=1e-8,
    weight_decay=1e-5
)

In [ ]:
epochs=100
# 🔹 Freeze backbone BEFORE training
for param in model.backbone.parameters():
    param.requires_grad = False

count = 0
patience = 5

best_f1 = 0.0
best_val_loss = float("inf")
min_delta = 1e-3

train_loss_list = []
val_loss_list = []
val_f1_macro_list = []
val_f1_weighted_list = []
val_acc_list = []
val_recons_loss = []
val_kl_div = []
val_classif_loss = []

for epoch in range(epochs):

    print(f"\nEpoch {epoch+1}")

    #  KL annealing
    beta = min(1.0, epoch / 20)

    # ---- TRAIN ----
    train_loss, train_cls, train_recon, train_kl = train_one_epoch(
        model,
        train_loader,
        device,
        loss_function,
        optimizer,
        epoch,
        epochs
    )

    # ---- VALIDATE ----
    val_loss, val_acc, val_f1_macro, val_f1_weighted, avg_r, avg_kl, avg_cl = validation(
        model,
        val_loader,
        device,
        loss_function,
        num_genres,
        threshold=0.2
    )

    # ---- STORE ----
    train_loss_list.append(train_loss)
    val_loss_list.append(val_loss)
    val_f1_macro_list.append(val_f1_macro)
    val_f1_weighted_list.append(val_f1_weighted)
    val_acc_list.append(val_acc)
    val_recons_loss.append(avg_r)
    val_kl_div.append(avg_kl)
    val_classif_loss.append(avg_cl)

    # ---- PRINT ----
    print(f"Train Loss: {train_loss:.4f} | CLS: {train_cls:.4f} | Recon: {train_recon:.4f} | KL: {train_kl:.4f}")
    print(f"Val Loss: {val_loss:.4f}")
    print(f"Val Acc: {val_acc:.4f}")
    print(f"Val Macro F1: {val_f1_macro:.4f}")
    print(f"Val Weighted F1: {val_f1_weighted:.4f}")
    print(f"Val Recon: {avg_r:.4f} | KL: {avg_kl:.4f} | CLS: {avg_cl:.4f}")

    # ---- EARLY STOPPING (IMPROVED LOGIC) ----
    if val_f1_macro > best_f1 + min_delta:
        best_f1 = val_f1_macro
        best_val_loss = val_loss
        count = 0

        torch.save(model.state_dict(), "/kaggle/working/checkpoint.pt")
        print(" Model improved, saving checkpoint...")

    else:
        count += 1
        print(f" No improvement. Patience: {count}/{patience}")

    if count >= patience:
        print(" Early stopping triggered...")
        break